In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from typing import List
from tqdm.notebook import tqdm  # notebook용 tqdm
import matplotlib.pyplot as plt

# PyTorch Geometric
from torch_geometric.nn import GATConv, GCNConv, SAGEConv

# Device 확인
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [2]:
def pairwise_distance_matrix(x_np: np.ndarray, device: str = 'cuda', batch_size: int = 1024) -> np.ndarray:
    """GPU-accelerated pairwise distance matrix computation."""
    N, D = x_np.shape
    dist_mat = torch.empty((N, N), dtype=torch.float32, device='cpu')

    x_norm_cpu = np.sum(x_np**2, axis=1)

    for i in range(0, N, batch_size):
        end_i = min(i + batch_size, N)
        x_batch = torch.from_numpy(x_np[i:end_i]).to(device).float()
        x_batch_norm = torch.from_numpy(x_norm_cpu[i:end_i]).to(device).float().unsqueeze(1)

        for j in range(0, N, batch_size):
            end_j = min(j + batch_size, N)
            x_ref = torch.from_numpy(x_np[j:end_j]).to(device).float()
            x_ref_norm = torch.from_numpy(x_norm_cpu[j:end_j]).to(device).float().unsqueeze(0)

            dot = x_batch @ x_ref.T
            dist_sq = x_batch_norm + x_ref_norm - 2 * dot
            dist_sq = torch.clamp(dist_sq, min=0.0)
            dist = torch.sqrt(dist_sq).cpu()

            dist_mat[i:end_i, j:end_j] = dist

    return dist_mat.numpy()


def distance_to_similarity(D: np.ndarray, method: str = 'gaussian') -> np.ndarray:
    """Convert distance matrix to similarity matrix [0, 1]."""
    if method == 'gaussian':
        sigma = np.median(D[D > 0])
        S = np.exp(-D**2 / (2 * sigma**2 + 1e-8))
    elif method == 'inverse':
        S = 1 / (1 + D)
    else:
        raise ValueError(f"Unknown method: {method}")
    
    S = np.clip(S, 0, 1)
    S = (S + S.T) / 2
    np.fill_diagonal(S, 1.0)
    
    return S

In [3]:
def compute_gain_matrix(D, medoids, nearest_dist, nearest_medoid, non_medoids):
    """Compute gain matrix for FasterPAM."""
    N, k = D.shape[0], medoids.shape[0]
    H = non_medoids.shape[0]

    D_h = D[:, non_medoids]
    gain_matrix = torch.zeros((H, k), device=D.device)
    old_total = nearest_dist.sum()

    for j in range(k):
        mask = torch.arange(k, device=D.device) != j
        alt_medoids = medoids[mask]
        D_alt = D[:, alt_medoids]
        alt_dist = D_alt.min(dim=1)[0]

        candidate_new_dists = torch.min(
            D_h.T.unsqueeze(2),
            alt_dist.unsqueeze(0).unsqueeze(2)
        ).squeeze(2)

        gain = old_total - candidate_new_dists.sum(dim=1)
        gain_matrix[:, j] = gain

    return gain_matrix


def fasterpam_gpu_vectorized(D: torch.Tensor, k: int, max_iter: int = 5, seed: int = 42) -> torch.Tensor:
    """FasterPAM algorithm on GPU."""
    torch.manual_seed(seed)
    N = D.shape[0]
    device = D.device

    medoids = torch.randperm(N, device=device)[:k]
    D_medoids = D[:, medoids]
    nearest_dist, nearest_medoid = D_medoids.min(dim=1)

    for it in range(max_iter):
        non_medoids = torch.tensor([i for i in range(N) if i not in medoids], device=device)
        gain_matrix = compute_gain_matrix(D, medoids, nearest_dist, nearest_medoid, non_medoids)

        best_h_idx, best_j = torch.nonzero(gain_matrix == gain_matrix.max(), as_tuple=True)
        best_h = non_medoids[best_h_idx[0]]

        if gain_matrix[best_h_idx[0], best_j[0]] <= 0:
            break

        medoids[best_j[0]] = best_h
        D_medoids = D[:, medoids]
        nearest_dist, nearest_medoid = D_medoids.min(dim=1)

    return medoids


def kmedoids_selection(data: np.ndarray, k: int, D: np.ndarray = None, device: str = 'cuda', seed: int = 42):
    """K-Medoids selection using FasterPAM."""
    random.seed(seed)
    np.random.seed(seed)
    
    if D is None:
        D = pairwise_distance_matrix(data, device=device)
    
    D_tensor = torch.from_numpy(D).to(device).float()
    medoid_indices = fasterpam_gpu_vectorized(D_tensor, k=k, max_iter=5, seed=seed)
    medoid_indices = medoid_indices.cpu().numpy()
    
    selection = np.zeros(len(data), dtype=bool)
    selection[medoid_indices] = True
    
    return selection, medoid_indices.tolist()

In [40]:
def compute_coverage(p, S, chunk_size=1000):
    """Coverage: sum_y [1 - prod_i (1 - p_i * S_iy)]"""
    n = len(p)
    device = p.device
    coverage = torch.tensor(0.0, device=device)
    
    for y_start in range(0, n, chunk_size):
        y_end = min(y_start + chunk_size, n)
        
        if isinstance(S, np.ndarray):
            S_chunk = torch.from_numpy(S[:, y_start:y_end]).float().to(device)
        else:
            S_chunk = S[:, y_start:y_end].to(device)
        
        p_S = p.unsqueeze(1) * S_chunk
        p_S = torch.clamp(p_S, max=0.9999)
        
        log_not_covered = torch.log(1 - p_S + 1e-10)
        prob_not_covered = log_not_covered.sum(dim=0).exp()
        
        coverage += (1 - prob_not_covered).sum()
    
    return coverage


def compute_diversity(p, S, chunk_size=1000):
    """Diversity: p^T S p"""
    n = len(p)
    device = p.device
    diversity = torch.tensor(0.0, device=device)
    
    for i_start in range(0, n, chunk_size):
        i_end = min(i_start + chunk_size, n)
        
        if isinstance(S, np.ndarray):
            S_chunk = torch.from_numpy(S[i_start:i_end, :]).float().to(device)
        else:
            S_chunk = S[i_start:i_end, :].to(device)
        
        p_chunk = p[i_start:i_end]
        diversity += (p_chunk * (S_chunk @ p)).sum()
    
    return diversity


def compute_total_loss(p, S, lambda_div=1.0, chunk_size=1000):
    """Total loss = -Coverage + lambda_div * Diversity + lambda_sparse * Sparsity"""
    coverage = compute_coverage(p, S, chunk_size)
    diversity = compute_diversity(p, S, chunk_size)
    
    loss = -coverage + lambda_div * diversity 
    
    return loss, coverage, diversity


def evaluate_selection(selection, S, device='cuda'):
    """Evaluate a binary selection."""
    p = torch.from_numpy(selection.astype(np.float32)).to(device)
    coverage = compute_coverage(p, S)
    diversity = compute_diversity(p, S)
    return coverage.item(), diversity.item()

In [ ]:
class DirectCoresetSelector:
    """Directly learn p without any model."""
    
    def __init__(self, n_samples, device='cuda'):
        self.n = n_samples
        self.device = device
        self.logits = torch.zeros(n_samples, device=device, requires_grad=True)
    
    def get_probabilities(self):
        return torch.sigmoid(self.logits)
    
    def get_selection(self, threshold=0.5):
        p = self.get_probabilities()
        return (p > threshold).cpu().numpy()
    
    def fit(self, S, lambda_div=1.0, lambda_sparse=0.01,
            lr=0.1, n_epochs=1000, chunk_size=1000, verbose=True):
        
        optimizer = torch.optim.Adam([self.logits], lr=lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs)
        
        history = {'loss': [], 'coverage': [], 'diversity': [], 'sparsity': [], 'n_selected': []}
        
        pbar = tqdm(range(n_epochs), disable=not verbose)
        for epoch in pbar:
            optimizer.zero_grad()
            
            p = self.get_probabilities()
            loss, coverage, diversity = compute_total_loss(
                p, S, lambda_div, chunk_size
            )
            
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            n_selected = (p > 0.5).sum().item()
            history['loss'].append(loss.item())
            history['coverage'].append(coverage.item())
            history['diversity'].append(diversity.item())
            history['n_selected'].append(n_selected)
            
            if verbose:
                pbar.set_description(
                    f"Loss: {loss.item():.2f} | Cov: {coverage.item():.2f} | "
                    f"Div: {diversity.item():.2f} | Sel: {n_selected}"
                )
        
        return history

In [42]:
def build_knn_graph(S: np.ndarray, k: int = 20):
    """Build k-NN graph from similarity matrix."""
    n = S.shape[0]
    edges = []
    weights = []
    
    for i in range(n):
        sims = S[i].copy()
        sims[i] = -1
        
        top_k_indices = np.argsort(sims)[-k:]
        
        for j in top_k_indices:
            if sims[j] > 0:
                edges.append([i, j])
                weights.append(sims[j])
    
    edge_index = torch.tensor(edges, dtype=torch.long).T
    edge_weight = torch.tensor(weights, dtype=torch.float)
    
    print(f"Graph: {n} nodes, {len(weights)} edges, Avg degree: {len(weights) / n:.1f}")
    
    return edge_index, edge_weight

In [43]:
class GATCoresetSelector(nn.Module):
    """Graph Attention Network for coreset selection."""
    
    def __init__(self, input_dim, hidden_dim=64, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        
        self.gat_layers = nn.ModuleList([
            GATConv(hidden_dim, hidden_dim, heads=n_heads, concat=False, dropout=dropout)
            for _ in range(n_layers)
        ])
        
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim) for _ in range(n_layers)
        ])
        
        self.output = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, edge_index, edge_weight=None):
        h = self.input_proj(x)
        
        for gat, norm in zip(self.gat_layers, self.layer_norms):
            h_new = gat(h, edge_index)
            h_new = self.dropout(F.relu(h_new))
            h = norm(h + h_new)
        
        logits = self.output(h).squeeze(-1)
        return torch.sigmoid(logits)


class GCNCoresetSelector(nn.Module):
    """Graph Convolutional Network for coreset selection."""
    
    def __init__(self, input_dim, hidden_dim=64, n_layers=3, dropout=0.1):
        super().__init__()
        
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        
        self.gcn_layers = nn.ModuleList([
            GCNConv(hidden_dim, hidden_dim) for _ in range(n_layers)
        ])
        
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim) for _ in range(n_layers)
        ])
        
        self.output = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, edge_index, edge_weight=None):
        h = self.input_proj(x)
        
        for gcn, norm in zip(self.gcn_layers, self.layer_norms):
            h_new = gcn(h, edge_index, edge_weight)
            h_new = self.dropout(F.relu(h_new))
            h = norm(h + h_new)
        
        logits = self.output(h).squeeze(-1)
        return torch.sigmoid(logits)

In [45]:
def plot_training_history(history, title="Training History"):
    """Plot training history."""
    fig, axes = plt.subplots(1, 4, figsize=(16, 3))
    
    axes[0].plot(history['loss'])
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Total Loss')
    axes[0].grid(True)
    
    axes[1].plot(history['coverage'], color='green')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Coverage')
    axes[1].set_title('Coverage')
    axes[1].grid(True)
    
    axes[2].plot(history['diversity'], color='red')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Diversity')
    axes[2].set_title('Diversity')
    axes[2].grid(True)
    
    axes[3].plot(history['n_selected'], color='purple')
    axes[3].set_xlabel('Epoch')
    axes[3].set_ylabel('# Selected')
    axes[3].set_title('Coreset Size')
    axes[3].grid(True)
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_probability_distribution(p, title="Selection Probabilities"):
    """Plot probability distribution."""
    if isinstance(p, torch.Tensor):
        p = p.detach().cpu().numpy()
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(p, bins=50, edgecolor='black', alpha=0.7)
    axes[0].axvline(x=0.5, color='red', linestyle='--', label='Threshold')
    axes[0].set_xlabel('Selection Probability')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'{title}: Distribution')
    axes[0].legend()
    
    sorted_p = np.sort(p)[::-1]
    axes[1].plot(sorted_p)
    axes[1].axhline(y=0.5, color='red', linestyle='--', label='Threshold')
    axes[1].set_xlabel('Sample Index (sorted)')
    axes[1].set_ylabel('Selection Probability')
    axes[1].set_title(f'{title}: Sorted')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()


def plot_comparison(results):
    """Plot comparison across methods."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    methods = list(set(results['method']))
    colors = plt.cm.tab10(np.linspace(0, 1, len(methods)))
    color_map = {m: c for m, c in zip(methods, colors)}
    
    for method in methods:
        mask = [m == method for m in results['method']]
        ks = [results['k'][i] for i, m in enumerate(mask) if m]
        
        objs = [results['objective'][i] for i, m in enumerate(mask) if m]
        covs = [results['coverage'][i] for i, m in enumerate(mask) if m]
        divs = [results['diversity'][i] for i, m in enumerate(mask) if m]
        
        marker = 'o' if method == 'K-Medoids' else '*'
        size = 80 if method == 'K-Medoids' else 150
        
        axes[0].scatter(ks, objs, label=method, marker=marker, s=size, c=[color_map[method]])
        axes[1].scatter(ks, covs, label=method, marker=marker, s=size, c=[color_map[method]])
        axes[2].scatter(ks, divs, label=method, marker=marker, s=size, c=[color_map[method]])
    
    axes[0].set_xlabel('Coreset Size (k)')
    axes[0].set_ylabel('Objective (lower is better)')
    axes[0].set_title('Objective vs K')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].set_xlabel('Coreset Size (k)')
    axes[1].set_ylabel('Coverage (higher is better)')
    axes[1].set_title('Coverage vs K')
    axes[1].legend()
    axes[1].grid(True)
    
    axes[2].set_xlabel('Coreset Size (k)')
    axes[2].set_ylabel('Diversity (lower is better)')
    axes[2].set_title('Diversity vs K')
    axes[2].legend()
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()

In [46]:

import sys
import os

project_root = '/data/basicts'  # 네 프로젝트 경로로 수정
sys.path.append(project_root)
os.chdir(project_root)


def load_traffic_data(dataset_name: str, train_val_test_ratio: List[float] = [0.6, 0.2, 0.2], 
                     input_len: int = 12, output_len: int = 12, mode: str = 'train'):
    """
    Load traffic time series data using TimeSeriesForecastingDataset.
    Expected shape: (n_timestamp, n_sensor, n_channel)
    Returns flattened: (n_sensor, n_timestamp * n_channel)
    
    각 센서를 하나의 샘플로 취급 (센서별 coreset selection)
    
    Args:
        dataset_name: 데이터셋 이름 (예: 'METR-LA', 'PEMS-BAY', 'ALAMEDA' 등)
        train_val_test_ratio: train/val/test 비율
        input_len: 입력 시퀀스 길이
        output_len: 출력 시퀀스 길이
        mode: 데이터 모드 ('train', 'valid', 'test')
    
    Returns:
        data_flat: (n_sensor, n_timestamp * n_channel) 형태의 flatten된 데이터
        original_shape: 원본 데이터 shape (n_timestamp, n_sensor, n_channel)
    """
    from basicts.data.simple_tsf_dataset import TimeSeriesForecastingDataset
    
    # 데이터셋 로드
    dataset = TimeSeriesForecastingDataset(
        dataset_name=dataset_name,
        train_val_test_ratio=train_val_test_ratio,
        mode=mode,
        input_len=input_len,
        output_len=output_len,
        memmap=False,  # numpy array로 로드 (coreset 계산을 위해)
    )
    
    # 데이터 가져오기: (n_timestamp, n_sensor, n_channel)
    return dataset

dataset = load_traffic_data("PEMS03")

Loading data from PEMS03


In [47]:
dataset_size = len(dataset)

mean = np.mean(dataset.data, axis=(0,1), keepdims=True)
std = np.std(dataset.data, axis=(0,1), keepdims=True)
std[std == 0] = 1.0

def transform(input_data):
    return (input_data - mean) / std

# Convert dataset into numpy array
inputs = np.array([
    transform(dataset[i]['inputs'])[:,:,[0,1,2]] +
    transform(dataset[i]['target'])[:,:,[1]]
    for i in range(dataset_size)])
inputs = inputs.reshape(dataset_size, -1)


distance_matrix = pairwise_distance_matrix(inputs, device=device)
similarity_matrix = distance_to_similarity(distance_matrix, method='gaussian')



In [48]:
# Hyperparameters
lambda_div = 1.0
lambda_sparse = 0.05
n_epochs = 500

# Direct P Learning
print("=" * 50)
print("Direct P Learning")
print("=" * 50)

direct_selector = DirectCoresetSelector(len(dataset), device=device)
history_direct = direct_selector.fit(
    similarity_matrix, 
    lambda_div=lambda_div, 
    lambda_sparse=lambda_sparse,
    n_epochs=n_epochs, 
    lr=0.1
)

# Results
direct_selection = direct_selector.get_selection()
direct_k = direct_selection.sum()
direct_cov, direct_div = evaluate_selection(direct_selection, S, device)

print(f"\nResults:")
print(f"  Selected: {direct_k}")
print(f"  Coverage: {direct_cov:.2f}")
print(f"  Diversity: {direct_div:.2f}")
print(f"  Objective: {-direct_cov + lambda_div * direct_div:.2f}")

# Visualize
plot_training_history(history_direct, "Direct P Learning")
plot_probability_distribution(direct_selector.get_probabilities(), "Direct P")

Direct P Learning


  0%|          | 0/500 [00:00<?, ?it/s]

NameError: name 'sparsity' is not defined